# Global Temperature Trends: Multi-Source EDA, Volatility, and Regional Warming

This notebook explores global surface temperature anomaly records from four widely used datasets:

Original GitHub repository: https://github.com/ByeonggeonGo/global_temperature_trend

- **NASA GISTEMP v4** global land-ocean temperature index
- **NOAA GlobalTemp v6.1** global land-ocean temperature anomalies
- **HadCRUT5** global monthly analysis with uncertainty intervals
- **Berkeley Earth High-Resolution** global monthly land-ocean series

The analysis is designed for a public GitHub/Kaggle workflow. It emphasizes reproducibility, dataset comparability, and clear communication of uncertainty.

## Main questions

1. How similar are the major global temperature datasets after rebasing them to a common reference period?
2. What is the long-term warming trend, and how does it change after 1970?
3. How much short-term variability sits on top of the warming signal?
4. Are recent years/months exceptional relative to the historical distribution?
5. Can we extend the analysis spatially with gridded data and GeoPandas?

## Important caveat

Temperature anomaly datasets use different native baselines. To compare them fairly, this notebook creates a common `1991-2020` anomaly scale. The original values are preserved, but most comparisons use the rebased values.

## 0. Environment and optional dependencies

The global time-series EDA uses `pandas`, `numpy`, `matplotlib`, and `scipy`. The regional map section additionally needs `xarray`, `netCDF4`, and `geopandas`.

If the geospatial packages are missing, uncomment and run the install cell below. Kaggle usually has many packages preinstalled, but local Windows environments often need the explicit install.

In [ ]:
# %pip install pandas numpy matplotlib scipy requests geopandas xarray netCDF4 shapely pyogrio

In [ ]:
from pathlib import Path
import json
import math
import re
import urllib.request
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from scipy.stats import linregress, zscore
except Exception:
    linregress = None
    zscore = None

warnings.filterwarnings("ignore", category=FutureWarning)

ROOT = Path.cwd()
DATA_DIR = ROOT / "temperature_dataset_samples"
OUT_DIR = ROOT / "analysis_outputs"
OUT_DIR.mkdir(exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (12, 6),
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.dpi": 110,
})

DATA_FILES = {
    "nasa": DATA_DIR / "nasa_gistemp_global_loti.csv",
    "noaa": DATA_DIR / "noaa_globaltemp_v6_1_land_ocean_global.asc",
    "hadcrut5": DATA_DIR / "hadcrut5_5_1_global_monthly.csv",
    "berkeley": DATA_DIR / "berkeley_earth_hr_global_monthly.txt",
}

missing = [str(path) for path in DATA_FILES.values() if not path.exists()]
if missing:
    raise FileNotFoundError("Missing input files. Run check_temperature_datasets.py first: " + ", ".join(missing))

## 1. Load and standardize the four global time series

Each loader returns a monthly table with:

- `dataset`
- `date`
- `year`
- `month`
- `anomaly_c`
- optional uncertainty columns when available

The files differ in format: NASA uses a wide annual CSV, NOAA uses whitespace-delimited ASCII, HadCRUT5 uses a tidy CSV, and Berkeley Earth uses a text file with metadata comments.

In [ ]:
def load_nasa_gistemp(path: Path) -> pd.DataFrame:
    lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
    header_at = next(i for i, line in enumerate(lines) if line.startswith("Year,"))
    df = pd.read_csv(path, skiprows=header_at)
    month_cols = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
    long = df.melt(id_vars="Year", value_vars=month_cols, var_name="month_name", value_name="anomaly_c")
    month_map = {name: i for i, name in enumerate(month_cols, start=1)}
    long["month"] = long["month_name"].map(month_map)
    long["anomaly_c"] = pd.to_numeric(long["anomaly_c"].replace("***", np.nan), errors="coerce")
    long["date"] = pd.to_datetime(dict(year=long["Year"].astype(int), month=long["month"], day=1))
    long["dataset"] = "NASA GISTEMP"
    return long[["dataset", "date", "anomaly_c"]].dropna()


def load_noaa_globaltemp(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, sep=r"\s+", header=None, comment="#")
    df = df.iloc[:, :3]
    df.columns = ["year", "month", "anomaly_c"]
    df["date"] = pd.to_datetime(dict(year=df["year"].astype(int), month=df["month"].astype(int), day=1))
    df["dataset"] = "NOAA GlobalTemp"
    return df[["dataset", "date", "anomaly_c"]].dropna()


def load_hadcrut5(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path)
    df["date"] = pd.to_datetime(df["Time"])
    df["anomaly_c"] = pd.to_numeric(df["Anomaly (deg C)"], errors="coerce")
    df["lower_95"] = pd.to_numeric(df["Lower confidence limit (2.5%)"], errors="coerce")
    df["upper_95"] = pd.to_numeric(df["Upper confidence limit (97.5%)"], errors="coerce")
    df["dataset"] = "HadCRUT5"
    return df[["dataset", "date", "anomaly_c", "lower_95", "upper_95"]].dropna(subset=["anomaly_c"])


def load_berkeley(path: Path) -> pd.DataFrame:
    rows = []
    for line in path.read_text(encoding="utf-8", errors="replace").splitlines():
        parts = line.split()
        if len(parts) >= 4 and parts[0].isdigit() and parts[1].isdigit():
            rows.append((int(parts[0]), int(parts[1]), float(parts[2]), float(parts[3])))
    df = pd.DataFrame(rows, columns=["year", "month", "anomaly_c", "uncertainty_95"])
    df["date"] = pd.to_datetime(dict(year=df["year"], month=df["month"], day=1))
    df["dataset"] = "Berkeley Earth HR"
    return df[["dataset", "date", "anomaly_c", "uncertainty_95"]]


series = pd.concat([
    load_nasa_gistemp(DATA_FILES["nasa"]),
    load_noaa_globaltemp(DATA_FILES["noaa"]),
    load_hadcrut5(DATA_FILES["hadcrut5"]),
    load_berkeley(DATA_FILES["berkeley"]),
], ignore_index=True, sort=False)

series["year"] = series["date"].dt.year
series["month"] = series["date"].dt.month
series["decimal_year"] = series["year"] + (series["month"] - 0.5) / 12

baseline_mask = series["year"].between(1991, 2020)
baseline = series[baseline_mask].groupby("dataset")["anomaly_c"].mean()
series["anom_1991_2020"] = series["anomaly_c"] - series["dataset"].map(baseline)

coverage = series.groupby("dataset").agg(
    start=("date", "min"),
    end=("date", "max"),
    months=("date", "count"),
    native_mean_1991_2020=("anomaly_c", lambda s: s[series.loc[s.index, "year"].between(1991, 2020)].mean()),
    missing_months=("anomaly_c", lambda s: s.isna().sum()),
).sort_index()
coverage

### EDA note: baseline compatibility

The table above is a quick audit. The native `1991-2020` mean differs because each dataset has its own original anomaly baseline. After subtracting that mean, all datasets are on a comparable `1991-2020 = 0` scale.

## 2. Annual aggregation, completeness checks, and core trend metrics

Annual means are computed from monthly values. Years with fewer than 10 available months are excluded to avoid partial-year distortions. This matters for the current year, which is usually incomplete.

In [ ]:
annual = (
    series.groupby(["dataset", "year"], as_index=False)
    .agg(
        anom_1991_2020=("anom_1991_2020", "mean"),
        native_anomaly_c=("anomaly_c", "mean"),
        months=("month", "count"),
    )
)
annual = annual[annual["months"] >= 10].copy()


def trend_per_decade(df: pd.DataFrame, start_year: int | None = None, end_year: int | None = None) -> dict:
    d = df.dropna(subset=["year", "anom_1991_2020"]).copy()
    if start_year is not None:
        d = d[d["year"] >= start_year]
    if end_year is not None:
        d = d[d["year"] <= end_year]
    x = d["year"].to_numpy(dtype=float)
    y = d["anom_1991_2020"].to_numpy(dtype=float)
    if len(d) < 3:
        return {"n_years": len(d), "trend_c_per_decade": np.nan, "p_value": np.nan, "r2": np.nan, "intercept": np.nan}
    if linregress is not None:
        fit = linregress(x, y)
        return {
            "n_years": len(d),
            "trend_c_per_decade": fit.slope * 10,
            "p_value": fit.pvalue,
            "r2": fit.rvalue ** 2,
            "intercept": fit.intercept,
        }
    slope, intercept = np.polyfit(x, y, 1)
    yhat = slope * x + intercept
    r2 = 1 - np.sum((y - yhat) ** 2) / np.sum((y - y.mean()) ** 2)
    return {"n_years": len(d), "trend_c_per_decade": slope * 10, "p_value": np.nan, "r2": r2, "intercept": intercept}

periods = {
    "full record": (None, None),
    "1950-present": (1950, None),
    "1970-present": (1970, None),
    "1990-present": (1990, None),
    "2000-present": (2000, None),
}

trend_rows = []
for dataset, group in annual.groupby("dataset"):
    for label, (start, end) in periods.items():
        trend_rows.append({"dataset": dataset, "period": label, **trend_per_decade(group, start, end)})

trends = pd.DataFrame(trend_rows).sort_values(["period", "dataset"])
trends[["dataset", "period", "n_years", "trend_c_per_decade", "r2", "p_value"]]

In [ ]:
fig, ax = plt.subplots(figsize=(13, 7))
for dataset, group in annual.groupby("dataset"):
    ax.plot(group["year"], group["anom_1991_2020"], linewidth=1.6, label=dataset)

ax.axhline(0, color="black", linewidth=0.9)
ax.set_title("Global annual temperature anomaly, rebased to 1991-2020")
ax.set_ylabel("Temperature anomaly (deg C)\nrelative to 1991-2020")
ax.set_xlabel("Year")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "global_annual_anomaly_comparison.png", dpi=180)
plt.show()

### Discussion: long-term trend signal

All four datasets tell the same broad story: global temperature has increased substantially since the nineteenth century, and the trend steepens after the mid-twentieth century. The exact slope differs slightly because the products use different interpolation methods, sea-surface temperature inputs, coverage assumptions, and uncertainty treatment.

The most policy-relevant visual feature is not a single record year. It is the sustained shift of the entire annual distribution upward. Once the records are rebased to `1991-2020`, the datasets track each other closely enough that the warming signal is robust to dataset choice.

## 3. Dataset agreement and divergence

A useful EDA check is to measure how far the datasets deviate from their multi-dataset mean. This helps reveal whether conclusions depend on a single source.

In [ ]:
wide_annual = annual.pivot(index="year", columns="dataset", values="anom_1991_2020")
wide_annual["multi_dataset_mean"] = wide_annual.mean(axis=1)
for dataset in [c for c in wide_annual.columns if c != "multi_dataset_mean"]:
    wide_annual[f"{dataset} residual"] = wide_annual[dataset] - wide_annual["multi_dataset_mean"]

agreement = []
for dataset in annual["dataset"].unique():
    residual = wide_annual[f"{dataset} residual"].dropna()
    agreement.append({
        "dataset": dataset,
        "mean_abs_residual_c": residual.abs().mean(),
        "max_abs_residual_c": residual.abs().max(),
        "residual_std_c": residual.std(),
        "correlation_with_multi_mean": wide_annual[[dataset, "multi_dataset_mean"]].corr().iloc[0, 1],
    })
agreement = pd.DataFrame(agreement).sort_values("mean_abs_residual_c")
agreement

In [ ]:
residual_cols = [c for c in wide_annual.columns if c.endswith(" residual")]
fig, ax = plt.subplots(figsize=(13, 6))
for col in residual_cols:
    ax.plot(wide_annual.index, wide_annual[col], linewidth=1.2, label=col.replace(" residual", ""))
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Dataset residuals relative to the multi-dataset annual mean")
ax.set_ylabel("Residual (deg C)")
ax.set_xlabel("Year")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "dataset_residuals_vs_multisource_mean.png", dpi=180)
plt.show()

### Discussion: agreement across independent products

The residual plot should be read as a method-sensitivity diagnostic. Larger early-period residuals are expected because sparse observations, especially over oceans and polar regions, make historical reconstruction harder. Modern residuals are smaller because observational coverage is denser and satellite-era constraints improve consistency.

For most headline conclusions, the four products are interchangeable. For uncertainty-aware statements, HadCRUT5 and Berkeley Earth are especially useful because they publish uncertainty information in the downloaded time-series products.

## 4. Volatility, short-term variability, and rolling trends

Long-term warming is not smooth. ENSO, volcanic eruptions, aerosol changes, ocean heat uptake, and internal variability create short-term fluctuations. This section separates trend-like movement from short-term volatility.

In [ ]:
monthly = series.sort_values(["dataset", "date"]).copy()
monthly["rolling_mean_120m"] = monthly.groupby("dataset")["anom_1991_2020"].transform(lambda s: s.rolling(120, min_periods=60).mean())
monthly["rolling_std_60m"] = monthly.groupby("dataset")["anom_1991_2020"].transform(lambda s: s.rolling(60, min_periods=36).std())
monthly["detrended_residual"] = monthly["anom_1991_2020"] - monthly["rolling_mean_120m"]

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
for dataset, group in monthly.groupby("dataset"):
    axes[0].plot(group["date"], group["rolling_mean_120m"], label=dataset)
    axes[1].plot(group["date"], group["rolling_std_60m"], label=dataset)

axes[0].set_title("10-year rolling mean")
axes[0].set_ylabel("Anomaly (deg C)")
axes[1].set_title("5-year rolling monthly volatility")
axes[1].set_ylabel("Standard deviation (deg C)")
axes[1].set_xlabel("Date")
axes[0].legend(ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "global_trend_and_volatility.png", dpi=180)
plt.show()

vol_summary = monthly.groupby("dataset").agg(
    monthly_std=("anom_1991_2020", "std"),
    detrended_std=("detrended_residual", "std"),
    latest_5yr_std=("rolling_std_60m", "last"),
)
vol_summary

In [ ]:
def rolling_trend(group: pd.DataFrame, window_years: int = 30) -> pd.DataFrame:
    d = group[["year", "anom_1991_2020"]].dropna().sort_values("year").copy()
    rows = []
    for end_year in range(int(d["year"].min()) + window_years - 1, int(d["year"].max()) + 1):
        w = d[d["year"].between(end_year - window_years + 1, end_year)]
        if len(w) >= window_years * 0.8:
            stats = trend_per_decade(w)
            rows.append({"dataset": group["dataset"].iloc[0], "end_year": end_year, "rolling_trend_c_per_decade": stats["trend_c_per_decade"]})
    return pd.DataFrame(rows)

rolling_trends = pd.concat([rolling_trend(g, 30) for _, g in annual.groupby("dataset")], ignore_index=True)

fig, ax = plt.subplots(figsize=(13, 6))
for dataset, group in rolling_trends.groupby("dataset"):
    ax.plot(group["end_year"], group["rolling_trend_c_per_decade"], label=dataset)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("30-year rolling warming trend")
ax.set_ylabel("deg C / decade")
ax.set_xlabel("End year of 30-year window")
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "rolling_30_year_trends.png", dpi=180)
plt.show()

### Discussion: variability versus climate trend

Short-term variability is large enough to create unusually warm or cool individual months, but it is much smaller than the multi-decadal shift in the rolling mean. The 30-year rolling trend view is useful because climate normals are typically discussed over multi-decadal windows. A persistent positive rolling trend across datasets is stronger evidence than any single anomalous month.

## 5. Seasonality, month-of-year behavior, and anomaly heatmaps

Global temperature anomalies are usually deseasonalized, but month-specific behavior can still differ because observational coverage, sea ice treatment, and circulation patterns vary through the year.

In [ ]:
monthly_climatology = monthly.groupby(["dataset", "month"]).agg(
    mean_anomaly=("anom_1991_2020", "mean"),
    std_anomaly=("anom_1991_2020", "std"),
    p05=("anom_1991_2020", lambda s: s.quantile(0.05)),
    p95=("anom_1991_2020", lambda s: s.quantile(0.95)),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5), sharex=True)
for dataset, group in monthly_climatology.groupby("dataset"):
    axes[0].plot(group["month"], group["mean_anomaly"], marker="o", label=dataset)
    axes[1].plot(group["month"], group["std_anomaly"], marker="o", label=dataset)
axes[0].set_title("Mean anomaly by calendar month")
axes[0].set_ylabel("Mean anomaly (deg C)")
axes[1].set_title("Volatility by calendar month")
axes[1].set_ylabel("Standard deviation (deg C)")
for ax in axes:
    ax.set_xlabel("Month")
    ax.set_xticks(range(1, 13))
axes[0].legend(fontsize=9)
fig.tight_layout()
fig.savefig(OUT_DIR / "month_of_year_eda.png", dpi=180)
plt.show()

monthly_climatology.head(12)

In [ ]:
heat_source = monthly[monthly["dataset"] == "NASA GISTEMP"].copy()
heat = heat_source.pivot_table(index="year", columns="month", values="anom_1991_2020", aggfunc="mean")
heat = heat.loc[1950:]

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(heat.values, aspect="auto", cmap="coolwarm", vmin=-1.5, vmax=1.5, origin="lower")
ax.set_title("NASA GISTEMP monthly anomaly heatmap, 1950-present")
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.set_xticks(np.arange(12))
ax.set_xticklabels(range(1, 13))
years = heat.index.to_numpy()
tick_idx = np.linspace(0, len(years) - 1, 9, dtype=int)
ax.set_yticks(tick_idx)
ax.set_yticklabels(years[tick_idx])
fig.colorbar(im, ax=ax, label="deg C relative to 1991-2020")
fig.tight_layout()
fig.savefig(OUT_DIR / "nasa_monthly_anomaly_heatmap.png", dpi=180)
plt.show()

## 6. Distribution shifts and extreme years/months

A warming climate shifts the whole anomaly distribution. This section compares earlier and later periods and lists the warmest years and months in each dataset.

In [ ]:
period_bins = [1850, 1900, 1950, 1980, 2000, 2015, 2100]
period_labels = ["1850-1899", "1900-1949", "1950-1979", "1980-1999", "2000-2014", "2015-present"]
monthly["period"] = pd.cut(monthly["year"], bins=period_bins, labels=period_labels, right=False)
period_summary = monthly.groupby(["dataset", "period"], observed=True).agg(
    mean=("anom_1991_2020", "mean"),
    median=("anom_1991_2020", "median"),
    std=("anom_1991_2020", "std"),
    p05=("anom_1991_2020", lambda s: s.quantile(0.05)),
    p95=("anom_1991_2020", lambda s: s.quantile(0.95)),
    months=("date", "count"),
).reset_index()
period_summary

In [ ]:
fig, ax = plt.subplots(figsize=(13, 6))
for dataset, group in period_summary.groupby("dataset"):
    ax.plot(group["period"].astype(str), group["mean"], marker="o", label=dataset)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("Distribution shift: mean monthly anomaly by historical period")
ax.set_ylabel("Mean anomaly (deg C)")
ax.set_xlabel("Period")
ax.tick_params(axis="x", rotation=20)
ax.legend(ncol=2)
fig.tight_layout()
fig.savefig(OUT_DIR / "period_distribution_shift.png", dpi=180)
plt.show()

In [ ]:
ranked_years = annual.copy()
ranked_years["warm_rank"] = ranked_years.groupby("dataset")["anom_1991_2020"].rank(ascending=False, method="min")
top_years = ranked_years[ranked_years["warm_rank"] <= 10].sort_values(["dataset", "warm_rank"])

ranked_months = monthly.copy()
ranked_months["warm_rank"] = ranked_months.groupby("dataset")["anom_1991_2020"].rank(ascending=False, method="min")
top_months = ranked_months[ranked_months["warm_rank"] <= 10].sort_values(["dataset", "warm_rank"])

print("Warmest annual anomalies by dataset")
display(top_years[["dataset", "warm_rank", "year", "anom_1991_2020", "months"]])

print("Warmest monthly anomalies by dataset")
display(top_months[["dataset", "warm_rank", "date", "anom_1991_2020"]])

### Discussion: extremes in context

The top-rank tables are useful for communication, but they should not be overinterpreted in isolation. In a warming series, record-breaking years become more likely because the baseline itself is rising. The more informative signal is that the recent period dominates the upper tail across independent datasets.

## 7. Uncertainty-aware view

HadCRUT5 provides lower and upper 95% confidence limits. Berkeley Earth provides a 95% uncertainty estimate in its text file. This section visualizes those intervals where available.

In [ ]:
had = monthly[monthly["dataset"] == "HadCRUT5"].dropna(subset=["lower_95", "upper_95"]).copy()
had_base = baseline.loc["HadCRUT5"]
had["lower_95_rebased"] = had["lower_95"] - had_base
had["upper_95_rebased"] = had["upper_95"] - had_base
had_annual = had.groupby("year", as_index=False).agg(
    anom_1991_2020=("anom_1991_2020", "mean"),
    lower_95_rebased=("lower_95_rebased", "mean"),
    upper_95_rebased=("upper_95_rebased", "mean"),
    months=("month", "count"),
)
had_annual = had_annual[had_annual["months"] >= 10]

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(had_annual["year"], had_annual["anom_1991_2020"], color="tab:red", label="HadCRUT5 annual mean")
ax.fill_between(
    had_annual["year"].to_numpy(),
    had_annual["lower_95_rebased"].to_numpy(),
    had_annual["upper_95_rebased"].to_numpy(),
    color="tab:red",
    alpha=0.18,
    label="Approximate annual 95% interval",
)
ax.axhline(0, color="black", linewidth=0.8)
ax.set_title("HadCRUT5 uncertainty interval, annual means")
ax.set_ylabel("deg C relative to 1991-2020")
ax.set_xlabel("Year")
ax.legend()
fig.tight_layout()
fig.savefig(OUT_DIR / "hadcrut5_uncertainty_interval.png", dpi=180)
plt.show()

### Discussion: uncertainty does not erase the trend

Uncertainty is widest in the nineteenth century, when observation networks were sparse. Even so, the modern warming signal is much larger than the historical uncertainty band. This is why independent products can differ in details while still agreeing on the overall warming trajectory.

## 8. Spatial EDA with gridded NOAA GlobalTemp and GeoPandas

The four downloaded global-average files do not contain latitude/longitude. For regional analysis, we need a gridded product.

This section downloads the latest NOAA GlobalTemp v6.1 gridded NetCDF file, computes each grid cell's 1970-present trend, and joins grid-cell trends to Natural Earth country polygons.

The first run downloads about 22 MB to `temperature_dataset_samples/noaa_globaltemp_gridded_latest.nc`.

In [ ]:
def latest_noaa_gridded_url() -> str:
    index_url = "https://www.ncei.noaa.gov/data/noaa-global-surface-temperature/v6.1/access/gridded/"
    html = urllib.request.urlopen(index_url, timeout=60).read().decode("utf-8", errors="replace")
    files = re.findall(r'NOAAGlobalTemp_v6\.1\.0_gridded_[^"<>]+\.nc', html)
    if not files:
        raise RuntimeError("No NOAA gridded NetCDF file found in the directory index")
    return index_url + sorted(set(files))[-1]


def download_if_missing(url: str, path: Path) -> Path:
    if path.exists() and path.stat().st_size > 1_000_000:
        return path
    print(f"Downloading {url}")
    with urllib.request.urlopen(url, timeout=180) as res:
        path.write_bytes(res.read())
    return path


gridded_url = latest_noaa_gridded_url()
gridded_path = download_if_missing(gridded_url, DATA_DIR / "noaa_globaltemp_gridded_latest.nc")
print(gridded_url)
print(gridded_path, f"{gridded_path.stat().st_size / 1_000_000:.1f} MB")

In [ ]:
import xarray as xr
import geopandas as gpd


ds = xr.open_dataset(gridded_path)
print(ds)

candidates = [name for name, da in ds.data_vars.items() if {"time", "lat", "lon"}.issubset(set(da.dims))]
if not candidates:
    candidates = [name for name, da in ds.data_vars.items() if len(da.dims) >= 3]
anom_var = next((name for name in candidates if "anom" in name.lower() or "temp" in name.lower()), candidates[0])
anom = ds[anom_var]

if "latitude" in anom.coords:
    anom = anom.rename({"latitude": "lat"})
if "longitude" in anom.coords:
    anom = anom.rename({"longitude": "lon"})
anom = anom.assign_coords(lon=(((anom["lon"] + 180) % 360) - 180)).sortby("lon")

annual_grid = anom.groupby("time.year").mean("time", skipna=True)
annual_grid = annual_grid.sel(year=slice(1970, None))
fit = annual_grid.polyfit(dim="year", deg=1, skipna=True)
slope_name = [v for v in fit.data_vars if v.endswith("polyfit_coefficients")][0]
trend_grid = fit[slope_name].sel(degree=1) * 10
trend_grid.name = "trend_c_per_decade_1970_present"

trend_df = trend_grid.to_dataframe().reset_index().dropna(subset=["trend_c_per_decade_1970_present"])
trend_df.head()

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
sc = ax.scatter(
    trend_df["lon"],
    trend_df["lat"],
    c=trend_df["trend_c_per_decade_1970_present"],
    s=12,
    cmap="inferno",
)
ax.set_title("NOAA GlobalTemp grid-cell warming trend, 1970-present")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
fig.colorbar(sc, ax=ax, label="deg C / decade")
fig.tight_layout()
fig.savefig(OUT_DIR / "noaa_gridcell_trends_scatter.png", dpi=180)
plt.show()

trend_df["lat_band"] = pd.cut(
    trend_df["lat"],
    bins=[-90, -60, -30, 0, 30, 60, 90],
    labels=["90S-60S", "60S-30S", "30S-0", "0-30N", "30N-60N", "60N-90N"],
)
trend_df["weight"] = np.cos(np.deg2rad(trend_df["lat"])).clip(lower=0)
zonal_trends = trend_df.groupby("lat_band", observed=True).apply(
    lambda g: np.average(g["trend_c_per_decade_1970_present"], weights=g["weight"])
).rename("area_weighted_trend_c_per_decade").reset_index()
zonal_trends

In [ ]:
world_url = "https://naturalearth.s3.amazonaws.com/110m_cultural/ne_110m_admin_0_countries.zip"
world = gpd.read_file(world_url).to_crs("EPSG:4326")

points = gpd.GeoDataFrame(
    trend_df,
    geometry=gpd.points_from_xy(trend_df["lon"], trend_df["lat"]),
    crs="EPSG:4326",
)
joined = gpd.sjoin(points, world[["ADMIN", "CONTINENT", "geometry"]], how="inner", predicate="within")
joined["weight"] = np.cos(np.deg2rad(joined["lat"])).clip(lower=0)


def weighted_mean(group: pd.DataFrame) -> float:
    return np.average(group["trend_c_per_decade_1970_present"], weights=group["weight"])

continent_trend = (
    joined.groupby("CONTINENT")
    .apply(weighted_mean)
    .rename("trend_c_per_decade_1970_present")
    .sort_values(ascending=False)
    .reset_index()
)

country_trend = (
    joined.groupby("ADMIN")
    .apply(weighted_mean)
    .rename("trend_c_per_decade_1970_present")
    .sort_values(ascending=False)
    .reset_index()
)

display(continent_trend)
display(country_trend.head(20))

In [ ]:
world_trend = world.merge(country_trend, left_on="ADMIN", right_on="ADMIN", how="left")

fig, ax = plt.subplots(figsize=(15, 7))
world_trend.plot(
    column="trend_c_per_decade_1970_present",
    ax=ax,
    legend=True,
    cmap="inferno",
    missing_kwds={"color": "lightgrey", "label": "No matched grid cells"},
    legend_kwds={"label": "deg C / decade"},
)
ax.set_title("NOAA GlobalTemp gridded trend by country, 1970-present")
ax.set_axis_off()
fig.tight_layout()
fig.savefig(OUT_DIR / "regional_country_trends_noaa_globaltemp.png", dpi=180)
plt.show()

### Discussion: regional warming

The gridded view usually reveals stronger warming over high northern latitudes than over the global mean. This is consistent with Arctic amplification and the general land-ocean contrast: land areas tend to warm faster than oceans.

Country-level values in this notebook are approximate. They are produced by assigning 5-degree grid-cell centers to Natural Earth country polygons and weighting by `cos(latitude)`. This is useful for EDA, but a production-grade regional climatology should use exact grid-cell polygon areas, masks for land/ocean fractions, and careful treatment of small island states and coastal cells.

## 9. Save derived tables and figures

The following outputs are useful for GitHub releases, Kaggle datasets, or downstream dashboards.

In [ ]:
annual.to_csv(OUT_DIR / "global_annual_anomalies_rebased_1991_2020.csv", index=False)
trends.to_csv(OUT_DIR / "global_trend_summary.csv", index=False)
vol_summary.to_csv(OUT_DIR / "global_volatility_summary.csv")
agreement.to_csv(OUT_DIR / "dataset_agreement_summary.csv", index=False)
period_summary.to_csv(OUT_DIR / "period_distribution_summary.csv", index=False)
top_years.to_csv(OUT_DIR / "top_warmest_years_by_dataset.csv", index=False)
top_months.to_csv(OUT_DIR / "top_warmest_months_by_dataset.csv", index=False)

if "zonal_trends" in globals():
    zonal_trends.to_csv(OUT_DIR / "regional_zonal_trends_noaa_globaltemp.csv", index=False)
if "continent_trend" in globals():
    continent_trend.to_csv(OUT_DIR / "regional_continent_trends_noaa_globaltemp.csv", index=False)
if "country_trend" in globals():
    country_trend.to_csv(OUT_DIR / "regional_country_trends_noaa_globaltemp.csv", index=False)

sorted(path.name for path in OUT_DIR.glob("*"))

## 10. Final interpretation

### What is robust?

- The four independent global datasets agree on a strong warming signal.
- The post-1970 trend is much steeper than the full-record trend because modern anthropogenic forcing dominates the recent period.
- Dataset differences are real but small relative to the multi-decadal warming signal.
- Short-term variability remains visible, especially month to month, but it does not reverse the long-term trend.
- Recent years dominate the warm tail of the distribution across datasets.

### What requires caution?

- Anomaly baselines must be harmonized before comparing values directly.
- The current year is usually incomplete and should not be ranked as a full year unless enough months are available.
- Regional estimates from coarse grids are exploratory. They are excellent for pattern discovery, but not a substitute for official country climate assessments.
- Berkeley Earth's current high-resolution global text product is marked preliminary by Berkeley Earth, so it is useful but should be cited with that caveat.

### Recommended next steps

1. Add ENSO indices to explain part of the short-term variability.
2. Compare land-only and ocean-only series to quantify the land-ocean warming contrast.
3. Use exact area-weighted masks for regional aggregation.
4. Add ERA5 or JRA-3Q reanalysis as a physically consistent spatial comparison.
5. Turn the derived tables into a small dashboard or Kaggle companion dataset.